# MonoTM: configure, check, run

Edit your data path, topic count, models, endpoint, and device below. The setup check reads your configuration and local environment without downloading models, training, or calling an LLM. Run the separate modeling cell when the displayed settings are right.

Install in this notebook's environment with `python -m pip install -e '.[embeddings]'` from the repository. The command-line alternative is `monotm config` → `monotm check` → `monotm run`. Paths below are relative to the working directory printed in the first cell; you can use absolute paths.


In [ ]:
from pathlib import Path
import logging
from monotm import check_run, run
from monotm.config import GenerationConfig, RunConfig
from monotm.preflight import format_report

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
print("Working directory:", Path.cwd())


## 1. Choose your settings

### Data and pipeline seed

Set the corpus paths and pipeline seed below. The next section configures the scikit-learn LDA estimator; MonoTM model and execution settings follow it.


In [ ]:
INPUT_CSV = Path("my_documents.csv")
OUTPUT_DIR = Path("runs/my_corpus")
TEXT_COLUMN = "text"
ID_COLUMN = None  # Set to your unique ID column, e.g. "document_id".
EMBEDDINGS = None  # Optional path to your .npy matrix.
EMBEDDING_IDS = None  # Optional JSON ID array for aligning embedding rows.

PIPELINE_SEED = 42  # SAE training and feature-example sampling.


### Scikit-learn LDA parameters

These parameters configure **`sklearn.decomposition.LatentDirichletAllocation` from scikit-learn**. MonoTM passes the `SKLEARN_LDA_PARAMS` dictionary to this estimator through `RunConfig.lda_params`.

Edit the values below. `n_components` sets the topic count throughout MonoTM. `PIPELINE_SEED` controls SAE training and example sampling; the LDA `random_state` uses that seed by default and can be changed separately. Review the complete resolved parameter list before fitting.

`learning_decay`, `learning_offset`, and `batch_size` apply to online learning. `perp_tol` requires positive `evaluate_every`. `total_samples` is used by scikit-learn’s `partial_fit`; this pipeline uses `fit_transform`, so that parameter has no effect here. See the [scikit-learn LDA reference](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html) for parameter details.


In [ ]:
# Scikit-learn LDA parameters (CPU).
# Parameter names match sklearn.decomposition.LatentDirichletAllocation.
SKLEARN_LDA_PARAMS = {
    "n_components": 20,          # Number of topics throughout MonoTM.
    "doc_topic_prior": None,    # Document-topic prior; None uses 1 / n_components.
    "topic_word_prior": None,   # Topic-feature prior; None uses 1 / n_components.
    "learning_method": "batch", # "batch" or "online" variational Bayes.
    "learning_decay": 0.7,      # Online learning rate decay; inactive in batch mode.
    "learning_offset": 10.0,    # Online learning offset; inactive in batch mode.
    "max_iter": 200,            # Maximum fitting iterations.
    "batch_size": 128,          # Online mini-batch size; inactive in batch mode.
    "evaluate_every": -1,       # Perplexity evaluation frequency; <= 0 disables it.
    "total_samples": 1e6,       # Used by sklearn partial_fit, not this fit workflow.
    "perp_tol": 0.1,            # Perplexity tolerance; used when evaluate_every > 0.
    "mean_change_tol": 0.001,   # Document-topic inference convergence tolerance.
    "max_doc_update_iter": 100, # Maximum document-topic inference iterations.
    "n_jobs": 1,                # Parallel CPU jobs; -1 uses all available CPUs.
    "verbose": 0,               # Logging verbosity.
    "random_state": PIPELINE_SEED, # LDA seed; None uses sklearn's random generator.
}


### MonoTM models and execution settings

`auto` selects CUDA when PyTorch detects it, otherwise CPU. `cpu` or `cuda:0` select a device explicitly. The embedding model and both SAEs run on that device; LDA and descriptor EM use CPU. Interpreter and Predictor run at the chosen endpoint, whose device is managed separately.

The defaults use local Nemotron embeddings and DeepInfra for the two LLM roles. Change the model IDs and endpoint together when using another provider. For local CPU execution, use `examples/cpu_smoke.json` and the chat server described in the README.

Each role has its own generation settings. Use `temperature=None` to omit temperature, `token_limit_parameter="max_completion_tokens"` for models requiring that field, and an optional supported `reasoning_effort`. `reasoning_effort=None` omits the field; the string `"none"` explicitly requests no reasoning. Token budgets using `max_completion_tokens` include reasoning tokens. The setup check displays the exact request fields and flags known incompatible settings.


In [ ]:
config = RunConfig(
    seed=PIPELINE_SEED,
    lda_params=SKLEARN_LDA_PARAMS,
    device="auto",
    embedding_model="nvidia/llama-embed-nemotron-8b",
    interpreter_model="google/gemma-3-27b-it",
    predictor_model="google/gemma-3-12b-it",
    interpreter_generation=GenerationConfig(temperature=0.0, token_limit=1024),
    predictor_generation=GenerationConfig(temperature=0.0, token_limit=1024),
    base_url="https://api.deepinfra.com/v1/openai",
    # max_features=20,  # Limit validation to 20 candidate features.
)


## 2. Review the effective setup

Check the three models, GPU/VRAM information, selected device, the complete scikit-learn LDA parameter list, inactive LDA options, and pipeline/LDA seeds, input count, output directory, API-key registration, and which stages will run or reuse cached outputs. A reported key is only a registration check; no secret value is displayed. Connectivity, model permissions/downloads, and whether a model fits GPU memory require actual execution.


In [ ]:
plan = check_run(
    INPUT_CSV, OUTPUT_DIR, config=config,
    text_column=TEXT_COLUMN, id_column=ID_COLUMN,
    embeddings_path=EMBEDDINGS, embedding_ids_path=EMBEDDING_IDS,
)
print(format_report(plan))


## 3. Register a key if needed

Prefer exporting `MONOTM_API_KEY` or `DEEPINFRA_API_KEY` before launching Jupyter. This optional cell uses hidden input instead of saving a token in notebook source. It does not contact the provider. The example loopback smoke server accepts a dummy key.


In [ ]:
import os
from getpass import getpass

if plan["llm"]["requests_needed"] and not plan["llm"]["key_present"]:
    os.environ["MONOTM_API_KEY"] = getpass("Chat endpoint API key: ")
    print("Key registered in this kernel; its value is hidden.")


## 4. Run with the current settings

This cell rechecks the current settings, then starts modeling. Document excerpts are sent to the endpoint shown in the plan. It reuses completed stages from a matching run directory; changed data/configuration may require a different output directory.


In [ ]:
plan = check_run(
    INPUT_CSV, OUTPUT_DIR, config=config,
    text_column=TEXT_COLUMN, id_column=ID_COLUMN,
    embeddings_path=EMBEDDINGS, embedding_ids_path=EMBEDDING_IDS,
)
print(format_report(plan))
if not plan["ready"]:
    raise RuntimeError("Fix the issues listed above before running.")

result = run(
    INPUT_CSV, OUTPUT_DIR, config=config,
    text_column=TEXT_COLUMN, id_column=ID_COLUMN,
    embeddings_path=EMBEDDINGS, embedding_ids_path=EMBEDDING_IDS,
)


## 5. Review topics and feature validation

For failed-response inspection, selected-feature model changes, manual labels, or Predictor-only retries, open [inspect_and_retry.ipynb](inspect_and_retry.ipynb). These repairs reuse trained SAEs and topic mixtures.


In [ ]:
print("Document-topic matrix:", result.theta.shape)
print("Topic-feature matrix:", result.beta.shape)
print("Outputs:", result.output_dir)
result.topics[:3]
